# Workstream 1: Dataset and segmentation baseline
Use this notebook to validate an approved local BraTS data setup and call reusable baseline components from `cap6411_atm`. Do not download or commit data or weights here. The dataset release, modalities, and split remain team decisions.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "cap6411_atm").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")
sys.path.insert(0, str(REPO_ROOT / "src"))

from cap6411_atm import PROJECT_ROOT
assert PROJECT_ROOT == REPO_ROOT
print(f"Project root: {PROJECT_ROOT}")

## Baseline training
The reusable loader, frozen DINOv3 segmentation model, and training pipeline now live under `src/cap6411_atm/`. The committed training configuration starts in deterministic pilot mode. Pilot metrics are explicitly marked and must not be reported as full-cohort results. The test split is not loaded during training or validation.

In [ ]:
from cap6411_atm.training import BaselineTrainingConfig

CONFIG_PATH = REPO_ROOT / "configs" / "brats2023_gli_baseline_training.json"
config = BaselineTrainingConfig.from_json(CONFIG_PATH)
assert config.pilot_enabled, "Run the pilot before enabling full-cohort training."
print(f"Scope: {config.scope}")
print(f"Batch size: {config.batch_size}; AMP: {config.amp}")
print(f"Archive: {config.archive_path}")
print(f"Manifest: {config.manifest_path}")

### Required first run: one-batch smoke test
Run the command below after mounting Drive and checking out the pinned official DINOv3 revision. It performs one optimizer update and reports throughput and peak CUDA allocation. Run pilot training as a separate process afterward so it starts from the official checkpoint.

In [ ]:
# Deliberately not executed in the repository notebook.
smoke_command = (
    "python -m cap6411_atm.training.baseline "
    "--config configs/brats2023_gli_baseline_training.json --smoke-test"
)
pilot_command = (
    "python -m cap6411_atm.training.baseline "
    "--config configs/brats2023_gli_baseline_training.json"
)
print(smoke_command)
print(pilot_command)